In [1]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from datetime import datetime

# Read data
data = pd.read_excel("Moderna_Pfizer Event Study.xlsx")


In [2]:
# Convert date
data['date'] = pd.to_datetime(data['Names Date'])

# Create Excess Returns
data['R_pfe'] = data['PFE'] - data['RF']
data['R_mrna'] = data['MRNA'] - data['RF']

# Training data for Pfizer
pfizer_data = data[(data['date'] >= pd.Timestamp('2019-01-01')) & (data['date'] <= pd.Timestamp('2020-12-03'))]

# Make sure Moderna has the equal training sample size
estim_window = len(pfizer_data)
moderna_end = pd.Timestamp('2020-12-10')
moderna_end_row = data[data['date'] == moderna_end].index[0]
moderna_start_row = moderna_end_row - estim_window + 1

# Training data for Moderna
moderna_data = data.iloc[moderna_start_row:moderna_end_row + 1]

# Run CAPM regressions
pfizer_capm = ols('R_pfe ~ Q("Mkt-RF")', data=pfizer_data).fit()
moderna_capm = ols('R_mrna ~ Q("Mkt-RF")', data=moderna_data).fit()

# Print the regression results
print("Pfizer CAPM Results:\n", pfizer_capm.summary())
print("\nModerna CAPM Results:\n", moderna_capm.summary())

Pfizer CAPM Results:
                             OLS Regression Results                            
Dep. Variable:                  R_pfe   R-squared:                       0.382
Model:                            OLS   Adj. R-squared:                  0.380
Method:                 Least Squares   F-statistic:                     298.7
Date:                Wed, 09 Apr 2025   Prob (F-statistic):           1.76e-52
Time:                        14:54:07   Log-Likelihood:                -859.45
No. Observations:                 486   AIC:                             1723.
Df Residuals:                     484   BIC:                             1731.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -0.0492      

In [3]:
# Define event windows
pfizer_event_start = pd.Timestamp('2020-12-04')
moderna_event_start = pd.Timestamp('2020-12-11')

# Filter event window data
pfizer_event_data = data[data['date'] >= pfizer_event_start]
moderna_event_data = data[data['date'] >= moderna_event_start]

# Calculate predicted returns using CAPM beta coefficients
pfizer_event_data['r_hat'] = pfizer_capm.predict(pfizer_event_data)
moderna_event_data['r_hat'] = moderna_capm.predict(moderna_event_data)

# Calculate abnormal returns
pfizer_event_data['ar'] = pfizer_event_data['R_pfe'] - pfizer_event_data['r_hat']
moderna_event_data['ar'] = moderna_event_data['R_mrna'] - moderna_event_data['r_hat']

# Print the first 20 AR's
print("First 20 AR's for Pfizer \n", pfizer_event_data['ar'].head(20))
print("First 20 AR's for Moderna \n", moderna_event_data['ar'].head(20))

First 20 AR's for Pfizer 
 486   -0.008317
487    2.285038
488    2.911139
489   -0.931154
490   -0.397762
491   -1.278972
492   -4.408710
493   -2.080728
494   -2.311759
495    0.023791
496   -0.790939
497   -0.593335
498   -1.702966
499    1.887738
500   -0.545047
501   -1.465351
502    0.941028
503   -0.967761
504   -0.020670
505    0.990851
Name: ar, dtype: float64
First 20 AR's for Moderna 
 491     0.258285
492    -1.710158
493    -5.845289
494    -7.521002
495     4.384502
496    -3.169459
497    -1.909504
498    -9.561666
499     2.955253
500    -5.938211
501   -10.364527
502     2.178950
503    -3.465834
504    -6.628767
505     6.611418
506    -2.995850
507     5.782730
508    -1.868850
509    -2.688834
510     3.513134
Name: ar, dtype: float64


/var/folders/vf/ml0jt6z91qsdhgg6jwg8p76h0000gn/T/ipykernel_31630/1754657862.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pfizer_event_data['r_hat'] = pfizer_capm.predict(pfizer_event_data)
/var/folders/vf/ml0jt6z91qsdhgg6jwg8p76h0000gn/T/ipykernel_31630/1754657862.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  moderna_event_data['r_hat'] = moderna_capm.predict(moderna_event_data)
/var/folders/vf/ml0jt6z91qsdhgg6jwg8p76h0000gn/T/ipykernel_31630/1754657862.py:14: SettingWithCopyWarning: 
A value

In [4]:
# Calculate cumulative abnormal returns
pfizer_event_data['car'] = np.cumsum(pfizer_event_data['ar'])
moderna_event_data['car'] = np.cumsum(moderna_event_data['ar'])

# Days of interest
days_of_interest = [4, 9, 14, 19]

# Extract CAR values
pfizer_CAR_values = pfizer_event_data['car'].iloc[days_of_interest]
moderna_CAR_values = moderna_event_data['car'].iloc[days_of_interest]

# Print the cumulative abnormal returns
print("Cumulative Abnormal Returns for Pfizer:")
for i, days in enumerate(days_of_interest):
    print(f"CAR for {days+1} days: {pfizer_CAR_values.iloc[i]:.3f}")

print("\nCumulative Abnormal Returns for Moderna:")
for i, days in enumerate(days_of_interest):
    print(f"CAR for {days+1} days: {moderna_CAR_values.iloc[i]:.3f}")

Cumulative Abnormal Returns for Pfizer:
CAR for 5 days: 3.859
CAR for 10 days: -6.197
CAR for 15 days: -7.942
CAR for 20 days: -8.464

Cumulative Abnormal Returns for Moderna:
CAR for 5 days: -10.434
CAR for 10 days: -28.057
CAR for 15 days: -39.726
CAR for 20 days: -37.984


/var/folders/vf/ml0jt6z91qsdhgg6jwg8p76h0000gn/T/ipykernel_31630/595311304.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pfizer_event_data['car'] = np.cumsum(pfizer_event_data['ar'])
/var/folders/vf/ml0jt6z91qsdhgg6jwg8p76h0000gn/T/ipykernel_31630/595311304.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  moderna_event_data['car'] = np.cumsum(moderna_event_data['ar'])
